# Exploring a global SCRIP grid with pyramids-mesh

**SCRIP** -- the *Spherical Coordinate Remapping and Interpolation Package* -- gives climate modelling its
standard way to describe *how to remap* a field from one grid to another. The SCRIP grid-description NetCDF is
the interchange format those remapping weights are built from: it stores a grid as a **list of cells by their
corners** (`grid_corner_lat`/`grid_corner_lon`), the cell centres (`grid_center_lat`/`lon`) and an active-cell
mask (`grid_imask`), with coordinates in **degrees**.

The file we open is a genuine production grid: a **global ~0.125 degree latitude/longitude grid** of
1441x2880 cells, the kind used to regrid satellite and climate-model fields. Stored in SCRIP's corner-list
convention it becomes an **unstructured mesh of ~4.15 million quadrilateral cells** -- a regular lat/lon raster
expressed as cells. `driver="scrip"` reads it into a pyramids `MeshDataset`, the same object every other
driver produces.

> Because the grid is four million cells, this notebook never draws it whole. We read it, inspect its global
> extent, then **crop** a small lon/lat window and do all the plotting and GIS there.

## 1. Imports

`MeshDataset` is the object the `scrip` driver hands back. Importing from `pyramids` also puts its vendored
GDAL/PROJ data in place, so nothing here is order-sensitive in a notebook -- just import the names you use.
`add_tiles` would add a web basemap *when a mesh is georeferenced*; a SCRIP grid records lon/lat with no CRS
object, so we only ever call it behind an `if ds.epsg` guard (which is why it never fires here).

In [ ]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
from pyramids.netcdf.ugrid import MeshDataset
from cleopatra.basemap.tiles import add_tiles

## 2. Locate the grid

This notebook lives in `docs/examples/real-data/`, so the shared `examples/data/real-data/` corpus is three
levels up. The SCRIP grid sits under `scrip/`. Paths inside a notebook are written **relative to the notebook
itself**, and pyramids' `read_file` takes a `pathlib.Path` directly -- we pass the `Path`, never `str(path)`.

In [ ]:
from pathlib import Path
DATA = Path("../../../examples/data/real-data")
grid = DATA / "scrip" / "1441x2880_SCRIP.20170901.nc"
grid, grid.exists()

## 3. Read the grid

`MeshDataset.read_file(path, driver="scrip")` dispatches the file to the `scrip` driver, which reads the
corner arrays and de-duplicates shared corners into a node/face table.

**Expect this read to be slow -- on the order of 20-30 seconds.** The file is ~382 MB and assembling
~4.15 million quadrilateral cells (with their shared-corner de-duplication) is real work; a wait of tens of
seconds here is normal for a grid this size, not a hang. We time it so the cost is explicit.

In [ ]:
t0 = time.perf_counter()
ds = MeshDataset.read_file(grid, driver="scrip")
read_seconds = time.perf_counter() - t0
print(f"read in {read_seconds:.1f} s")
ds

A ~382 MB SCRIP grid reads into a 2-D `MeshDataset` in tens of seconds: roughly **4.15 million nodes and
4.15 million faces**, and no data variables -- a SCRIP grid is a pure geometry description, not a field
dataset. (Node and face counts are nearly equal because a regular lat/lon grid has about as many shared
corners as cells.)

## 4. Choose the plotting backend

cleopatra switches matplotlib to a headless `Agg` backend on the first `read_file`. Immediately after that
first read we call cleopatra's own chooser once: it detects the notebook and selects the inline backend (and
falls back to `Agg` in a plain script or in CI), so every figure below embeds natively with no manual
rasterising.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## 5. Explore the dataset

A `MeshDataset` exposes summary properties describing the whole mesh at a glance: `n_node` / `n_face` (size),
`crs` / `epsg` (georeferencing, if the format recorded any), `bounds` (spatial extent), and
`data_variable_names` (any attached fields).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(b, 4) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

`crs`/`epsg` are `None` -- SCRIP records lon/lat with no CRS object, so reprojection and basemap tiles are off
the table until a CRS is assigned. The `bounds` tell the real story: longitude spans ~`-0.06 -> 359.94` and
latitude the full `-90 -> 90`. That is a **global** grid, and the numbers are plainly in **degrees**.

## 6. The coordinates are global degrees

Reading the node coordinates straight off the `Mesh2d` confirms the extent in both axes. A regular ~0.125
degree grid places its first cell centre near the prime meridian, so the corner longitudes start just west of
zero and wrap almost all the way to 360; latitudes reach both poles.

In [ ]:
nx, ny = ds.mesh.node_x, ds.mesh.node_y
pd.Series({
    "lon min/max": (round(float(nx.min()), 3), round(float(nx.max()), 3)),
    "lat min/max": (round(float(ny.min()), 3), round(float(ny.max()), 3)),
})

Longitude runs the full `0 -> 360` band and latitude the full `-90 -> 90` -- a truly global grid, in degrees.
Once a geographic CRS is assigned these are already in the units most GIS tools expect, with no radian
conversion.

## 7. Quadrilateral cells

`face_node_connectivity` is the `(n_face, k)` table listing the node indices of each face, padded with `-1`
where a face has fewer than `k` corners. For a lat/lon grid every cell is a four-cornered quad, so we expect
a fixed width of exactly four and no padding.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
widths = (conn >= 0).sum(axis=1)
pd.Series({
    "connectivity_shape": conn.shape,
    "nodes_per_face (unique)": np.unique(widths).tolist(),
})

The table is `(n_face, 4)` and **every** face uses exactly four nodes -- all quadrilaterals, no triangles,
no padding. These are the familiar lat/lon cells, each bounded by two meridians and two parallels.

## 8. Four million cells is too large to draw

At ~4.15 million faces, `plot_outline()` or a whole-grid data map would try to render millions of polygons --
minutes of drawing, and an unreadable solid-black rectangle at the end. A global wireframe is the wrong tool
at this size. The right move is the same one you would use on a huge raster: **crop a small window first** and
work there.

## 9. Crop a small lon/lat window

`crop(bbox=(xmin, ymin, xmax, ymax))` returns a new `MeshDataset` holding only the faces inside the box -- the
mesh equivalent of a raster window. Because the coordinates are **degrees**, the box is given in degrees too.
We take a ~5 degree x 5 degree mid-latitude window (10-15 degrees E, 40-45 degrees N) and confirm how many
faces it keeps before plotting anything.

In [ ]:
bbox = (10.0, 40.0, 15.0, 45.0)   # lon 10-15 E, lat 40-45 N -- in degrees
cropped = ds.crop(bbox=bbox)
print(f"faces: {ds.n_face:,} -> {cropped.n_face:,}")
cropped.n_face

The crop drops ~4.15 million faces down to a handful of thousand -- a 5 degree x 5 degree window at ~0.125
degree spacing is about 40 x 40 cells, which is exactly the order of magnitude we get. That is a size a
plotter and the GIS layer can handle instantly.

In [ ]:
_g = cropped.plot_outline()
if cropped.epsg:
    add_tiles(_g.ax, crs=cropped.epsg)
_g.ax.set_title("SCRIP grid -- cropped 5 x 5 degree window")
_g.ax.set_aspect("equal")

The wireframe reveals the structure the four-million-cell view hid: a perfectly regular lattice of uniform
square cells. Locally, the global SCRIP grid is just a regular lat/lon raster drawn as cells.

## 10. A data map on the subset

A SCRIP grid carries no field values, so we synthesise one on the subset. We attach each cell's **centroid
latitude** as a face-located variable with `with_variable(...)`, which returns a **new** dataset (the original
is left untouched). Colouring by it produces the generic pyramids data-map that any real per-cell field
(temperature, flux, remapping weight, ...) would use.

In [ ]:
cx, cy = cropped.mesh.face_centroids
d2 = cropped.with_variable("cell_lat", cy, location="face")
_g = d2.plot("cell_lat", cmap="viridis", colorbar=True, title="Cell centroid latitude (degrees N)")
if d2.epsg:
    add_tiles(_g.ax, crs=d2.epsg)
_g.ax.set_aspect("equal")

The map is a clean sequence of horizontal colour bands climbing from 40 to 45 degrees -- each row of cells
shares one latitude, the visual signature of a regular lat/lon grid. `summary()` is the numeric companion:

In [ ]:
d2.summary()

The single `cell_lat` row spans `40 -> 45` with a mean of `42.5`, matching the window we cropped.

One more teaching point about these cells. Each cell's **planar** area (in degree-squared units) is identical
across the whole window -- the angular step is constant, which is precisely what *regular* lat/lon means.

In [ ]:
areas = cropped.mesh.face_areas
pd.Series({
    "unique_face_areas_deg2": np.unique(np.round(areas, 6)).tolist(),
    "step_0.125_squared": round(0.125 ** 2, 6),
})

Every cell measures `0.015625` degree-squared = `0.125 ** 2`, confirming a uniform 0.125 degree spacing. Note
the subtlety: the constant value is the *angular* (planar degree-squared) area. The true area **on the
sphere** shrinks toward the poles by a factor of `cos(latitude)` -- the grid keeps a constant angular step, not
a constant metric area, which is exactly why lat/lon cells look stretched near the poles.

## 11. GIS operations, for free

Because the mesh is a pyramids object, the pyramids GIS layer applies directly to the cropped subset -- none
of these methods live in `pyramids-mesh`, which only reads the file. We tour three.

### Faces as a `GeoDataFrame`

`to_geodataframe()` turns every face into a polygon row of a geopandas `GeoDataFrame` -- the standard vector
structure for export to shapefile/GeoPackage or for spatial joins.

In [ ]:
gdf = cropped.to_geodataframe()
print(type(gdf).__name__, gdf.shape)
gdf.head()

One quadrilateral polygon per cell. The CRS is unset (inherited `None`), so you would assign a geographic CRS
before exporting.

### The outer boundary

`bounding_polygon()` dissolves the cropped cells into their single enclosing polygon -- here the rectangle of
the lon/lat window we cut.

In [ ]:
poly = cropped.bounding_polygon()
ax = gpd.GeoSeries([poly]).plot(facecolor="none", edgecolor="red", linewidth=2)
if cropped.epsg:
    add_tiles(ax, crs=cropped.epsg)
ax.set_aspect("equal")
ax.set_title("cropped window -- bounding polygon")
pd.Series({"geom_type": poly.geom_type, "bounds": tuple(round(b, 3) for b in cropped.bounds)})

A single rectangular `Polygon` snug around the window -- an ordinary shapely geometry, ready for clipping,
buffering or a spatial join.

### The face-adjacency graph

`sparse_adjacency()` returns the face-to-face adjacency as a SciPy sparse matrix, and `connected_components()`
labels which faces reach which through shared edges. A contiguous patch of grid is one single component -- the
quickest topological health check.

In [ ]:
adj = cropped.sparse_adjacency()
labels = cropped.connected_components()
pd.Series({
    "adjacency_shape": adj.shape,
    "adjacency_nnz": int(adj.nnz),
    "connected_components": int(np.unique(labels).size),
})

**One** connected component: the cropped window is a single edge-connected patch of the grid, with every cell
wired to its neighbours -- exactly what a clean regular grid should report.

> **Note -- xarray interop.** pyramids-core exposes a `to_xarray()` bridge to the xarray ecosystem, but these
> notebooks deliberately stay within pyramids-core's own GDAL-based UGRID I/O and do not import `xarray` (nor
> `h5py`, `netCDF4`, or `scipy`) directly.

## Takeaway

- `driver="scrip"` reads a SCRIP remapping grid (corner-list convention) into a pyramids `MeshDataset`,
  de-duplicating shared corners into nodes -- here a **global ~0.125 degree grid of ~4.15 million
  quadrilateral cells**.
- The read is genuinely slow (tens of seconds) and the grid is far too large to draw whole; the right
  workflow is to **`crop(bbox=...)` a small window in degrees** and plot and analyse there.
- Coordinates are geographic **degrees** (`lon 0-360`, `lat -90-90`) with no CRS recorded, and the grid is
  geometry-only (no data variables), so we synthesised a per-cell `cell_lat` field to map it.
- Locally the SCRIP grid is a **regular lat/lon raster expressed as cells**: uniform quads, constant angular
  area -- while the full pyramids GIS layer (`to_geodataframe`, `bounding_polygon`, `connected_components`)
  applies to the cropped subset for free.